<a href="https://colab.research.google.com/github/OsaidKamran/FLYRANK.AI-SUMMER-INTERNSHIP-MACHINE-LEARNING_UPDATED/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OsaidKamran/FLYRANK.AI-SUMMER-INTERNSHIP-MACHINE-LEARNING_UPDATED/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row represents one pseudonymized content_id's aggregated performance signature for month=2026-03, pooled across all clients in the warehouse. The underlying warehouse table's native grain is finer than this — one row per report_date, client_id, and content_id — so this contract's row is a deliberate aggregation up from that daily grain, not the raw table's own grain.

The time window is month=2026-03, a mid-panel month chosen deliberately over fact_content_daily_performance_sample, which is the sealed final month of June 2026. Using the sample table to develop label logic would mean building inside the natural outcome window of any past-to-future label, which is exactly the trap this internship's data guide warns against. Within the month, features are built from the first half of the month, days 1 through 15, and the label comparison comes from the second half, days 16 through 31, so that every feature is provably knowable before the window the label is drawn from.

Content items are pooled across clients rather than analyzed within a single client, because this project's goal of discovering performance archetypes across FlyRank's content inventory requires patterns that generalize beyond any one account. client_id is therefore never a feature; it is retained only as a grouping and joining key.

In [8]:
# ==============================================================================
# ML-04: SEARCH INTELLIGENCE DATA CONTRACT & LEAKAGE AUDIT
# ==============================================================================
import os
import pandas as pd
import duckdb
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from huggingface_hub import list_repo_files

# ===== STEP 0: ENVIRONMENT SETUP & SCHEMA INSPECTION =====
print("===== STEP 0: ENVIRONMENT SETUP & SCHEMA INSPECTION =====")

# Safely acquire Hugging Face Token (Colab Secrets -> getpass fallback)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("✓ Successfully retrieved HF_TOKEN from Colab secrets.")
except (ImportError, Exception):
    import getpass
    print("Colab secrets not detected. Please provide your Hugging Face READ token:")
    HF_TOKEN = getpass.getpass("HF_TOKEN: ")

# Connect to DuckDB and configure HTTPFS for Hugging Face
try:
    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs;")
    con.execute(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');")
    print("✓ DuckDB configured and Hugging Face secret mounted.")
except Exception as e:
    raise RuntimeError(f"Failed to configure DuckDB or authenticate with Hugging Face: {e}")

# Fetch and analyze actual repository structure from Hugging Face
print("\nFetching actual repository structure from Hugging Face...")
try:
    repo_files = list_repo_files(repo_id="FlyRank/internship-warehouse", repo_type="dataset", token=HF_TOKEN)

    print(f"Total files in repo: {len(repo_files)}")

    # Programmatically find the true base files for our target tables by searching the file list
    client_file = next((f for f in repo_files if 'dim_clients' in f and f.endswith('.parquet')), None)
    content_file = next((f for f in repo_files if 'dim_content' in f and f.endswith('.parquet')), None)
    daily_files = [f for f in repo_files if 'fact_content_daily_performance' in f and '2026-03' in f and f.endswith('.parquet')]

    # Explicitly check against None instead of truthiness to allow root files (dirname == "")
    if client_file is None or content_file is None or not daily_files:
        raise FileNotFoundError("Could not dynamically locate all required tables or the month=2026-03 partition in the repo listing.")

    # Construct correct paths handling root-level files vs subfolders
    if os.path.dirname(client_file) == "":
        client_path = f"hf://datasets/FlyRank/internship-warehouse/{client_file}"
    else:
        client_path = f"hf://datasets/FlyRank/internship-warehouse/{os.path.dirname(client_file)}/**/*.parquet"

    if os.path.dirname(content_file) == "":
        content_path = f"hf://datasets/FlyRank/internship-warehouse/{content_file}"
    else:
        content_path = f"hf://datasets/FlyRank/internship-warehouse/{os.path.dirname(content_file)}/**/*.parquet"

    # For daily performance, check naming convention and apply correct glob/path
    daily_dir = os.path.dirname(daily_files[0])
    if daily_dir == "":
        daily_path = f"hf://datasets/FlyRank/internship-warehouse/{daily_files[0]}"
    else:
        daily_path = f"hf://datasets/FlyRank/internship-warehouse/{daily_dir}/**/*.parquet"

    print(f"Resolved client_path:  {client_path}")
    print(f"Resolved content_path: {content_path}")
    print(f"Resolved daily_path:   {daily_path}")

except Exception as e:
    raise RuntimeError(f"Failed to fetch repository file listing or resolve paths: {e}")

# Inspect schema dynamically to confirm real column names
print("\nInspecting Schema directly from Parquet files...")
clients_schema = con.execute(f"DESCRIBE SELECT * FROM '{client_path}'").df()
content_schema = con.execute(f"DESCRIBE SELECT * FROM '{content_path}'").df()
daily_schema = con.execute(f"DESCRIBE SELECT * FROM '{daily_path}'").df()

d_cols = daily_schema['column_name'].tolist()
c_cols = content_schema['column_name'].tolist()

print(f"-> dim_clients columns found: {clients_schema['column_name'].tolist()}")
print(f"-> dim_content columns found: {c_cols}")
print(f"-> fact_content_daily_performance columns found: {d_cols}")

# Confirmed real schema column assignments (no guesses/fallbacks)
col_imp = 'gsc_impressions'
col_clk = 'gsc_clicks'
col_content_id = 'content_hash_id'
col_client_id = 'client_hash_id'
col_date = 'content_created_date'

print("\nConfirmed Real Column Mapping Applied:")
print(f"  - Content ID: {col_content_id}")
print(f"  - Client ID: {col_client_id}")
print(f"  - Engagement Rate: Derived Expression (ga4_engaged_sessions / NULLIF(ga4_sessions, 0))")
print(f"  - Content Creation Date: {col_date}")

===== STEP 0: ENVIRONMENT SETUP & SCHEMA INSPECTION =====
Colab secrets not detected. Please provide your Hugging Face READ token:
HF_TOKEN: ··········
✓ DuckDB configured and Hugging Face secret mounted.

Fetching actual repository structure from Hugging Face...
Total files in repo: 24
Resolved client_path:  hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet
Resolved content_path: hf://datasets/FlyRank/internship-warehouse/dim_content.parquet
Resolved daily_path:   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/**/*.parquet

Inspecting Schema directly from Parquet files...
-> dim_clients columns found: ['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']
-> dim_content columns found: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'co

## 2. Fields: feature / label / context / excluded

impressions_first_half, avg_position_first_half, derived_ctr_first_half, engagement_rate_first_half, and content_age_days are all features, because each is computed entirely from the first half of the month and is fully observed before the target window begins. engagement_rate_first_half is additionally filtered to rows where ga4_data_available is true, so it never draws on the zero-filled placeholder values that appear before a client's GA4 tracking began.

avg_position_second_half is the one label-derived column in this notebook. It is used only to construct the proxy label, position_improved, and is deliberately excluded from the honest feature set. It is the column intentionally reintroduced later to demonstrate leakage. position_improved itself is the label: a binary flag equal to one if avg_position_second_half is numerically lower than avg_position_first_half within month=2026-03.

content_id and client_id are both context fields. content_id is the grouping key this project's unit of analysis is built on. client_id is used only to confirm the pooling assumption holds, checked directly in Section 3's grain query, and is never passed into any model.

Two things are excluded outright. The fact_content_query_90d table is excluded because it is query-hash and semantic-adjacent data, which falls outside a lane that deliberately avoids semantic-clustering claims. Any product-decision flag, such as a health-score-style column, would also be excluded, since learning from it risks circular machine learning: the model would be learning FlyRank's own internal decision rather than an observable signal.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

Three claims from the sections above are verified with queries, followed by a five-feature frame and a deliberate leakage demonstration.

The grain check confirms that content_id is safe to use as the sole grouping key under the pooling assumption stated in Section 1: zero content_ids were found under more than one client_id in month=2026-03, out of 9,841,378 rows checked.

The row count and date span check shows month=2026-03 contains 9,841,378 total rows in fact_content_daily_performance. The global date span runs from March 1 to March 31, 2026, the full calendar month as expected. Per-client spans vary considerably, however: most clients show the full 30-day span, but several show much shorter windows, down to as little as 8 days for the client with the narrowest history. This confirms the panel-depth warning directly: a global date range alone would have hidden these truncated clients inside an otherwise clean-looking month.

The availability check shows that filtering on ga4_data_available equal to true retains only 413,966 of 9,841,378 total rows, or 4.21 percent. This is a substantially more severe gap than a client-level view would suggest: a client can hold GA4 access broadly while still having most individual daily rows flagged unavailable, so the row-level survival rate is the more honest measure of how much engagement-based analysis this slice can actually support.

The five-feature frame was built at the content_id grain using only days 1 through 15 of the month, with each feature's availability justification printed inline in the code output. Rows with zero first-half impressions were dropped before modeling, leaving 151,981 feature rows.

For the leakage trap, an honest baseline model trained on only the five first-half features scored 0.7033 ROC-AUC. Deliberately adding avg_position_second_half, the exact column the label is derived from, as a sixth input pushed the score to 0.9272 ROC-AUC, an artificial jump toward perfect. This column is leaky because it is not an independent predictor of the outcome; it is algebraically part of how the outcome was defined in the first place, so the model was reading the answer key rather than learning a pattern. Removing it and retraining recovered a score of 0.7033 ROC-AUC, exactly matching the original honest baseline and confirming the jump was caused entirely by the injected leak.

In [10]:
# ===== PART A: THREE VERIFICATION QUERIES =====
print("\n===== PART A: THREE VERIFICATION QUERIES =====")

# 1. Grain Check
print("\n--- 1. Grain Check ---")
q_grain = f"""
SELECT {col_content_id}, COUNT(DISTINCT {col_client_id}) as distinct_clients
FROM '{daily_path}'
GROUP BY {col_content_id}
HAVING COUNT(DISTINCT {col_client_id}) > 1
"""
grain_df = con.execute(q_grain).df()
duplicate_groups = grain_df.shape[0]

print(f"Number of {col_content_ids} appearing under more than one {col_client_id}: {duplicate_groups}" if 'col_content_ids' in locals() else f"Number of {col_content_id} appearing under more than one {col_client_id}: {duplicate_groups}")
if duplicate_groups == 0:
    print(f"-> Confirmed: {col_content_id} is a safe, globally unique identifier across clients. It can be safely used as the sole grouping key for this project's pooled-across-clients unit of analysis.")
else:
    print(f"-> WARNING: {col_content_id} is NOT globally unique. The true grain requires ({col_client_id}, {col_content_id}). Sample offending counts:")
    print(grain_df[[col_content_id, 'distinct_clients']].head().to_string(index=False))

# 2. Row Count + Date Span
print("\n--- 2. Row Count & Date Span ---")
q_span = f"""
SELECT
    COUNT(*) as total_rows,
    MIN(report_date) as global_min_date,
    MAX(report_date) as global_max_date
FROM '{daily_path}'
"""
span_df = con.execute(q_span).df()
print(f"Total row count for month=2026-03: {span_df['total_rows'][0]:,}")
print(f"Global date span: {span_df['global_min_date'][0]} to {span_df['global_max_date'][0]}")

q_client_span = f"""
SELECT
    {col_client_id},
    MIN(report_date) as min_date,
    MAX(report_date) as max_date,
    DATE_DIFF('day', CAST(MIN(report_date) AS DATE), CAST(MAX(report_date) AS DATE)) as span_days
FROM '{daily_path}'
GROUP BY {col_client_id}
ORDER BY span_days DESC
"""
client_span_df = con.execute(q_client_span).df()
print(f"\nDate span grouped by {col_client_id} (Sample of widest and narrowest spans):")
print(pd.concat([client_span_df.head(5), client_span_df.tail(5)]).to_string(index=False))
print("-> Confirmed: History depth varies heavily by client. Global limits obscure truncated client data.")

# 3. Availability Check
print("\n--- 3. Availability Check (GA4 Zero-Fill Audit) ---")
q_avail = f"""
SELECT
    COUNT(*) as total_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as valid_ga4_rows
FROM '{daily_path}'
"""
avail_df = con.execute(q_avail).df()
total_rows = avail_df['total_rows'][0]
valid_rows = avail_df['valid_ga4_rows'][0]
survival_pct = (valid_rows / total_rows) * 100 if total_rows > 0 else 0
print(f"Total rows before filter: {total_rows:,}")
print(f"Total rows after 'ga4_data_available IS TRUE' filter: {valid_rows:,}")
print(f"Survival percentage: {survival_pct:.2f}%")


# ===== PART B: FIVE-FEATURE FRAME =====
print("\n===== PART B: FIVE-FEATURE FRAME =====")
# Note: Constructing features strictly using days 1-15 to leave days 16-31 isolated for our target window.
q_features = f"""
SELECT
    d.{col_content_id},

    -- Feature 1: Trailing Impressions (Days 1-15)
    SUM(CASE WHEN EXTRACT(DAY FROM CAST(d.report_date AS DATE)) <= 15 THEN CAST(d.{col_imp} AS FLOAT) ELSE 0 END) as impressions_first_half,

    -- Feature 2: Average Position (Days 1-15)
    AVG(CASE WHEN EXTRACT(DAY FROM CAST(d.report_date AS DATE)) <= 15 AND d.gsc_avg_position > 0 THEN CAST(d.gsc_avg_position AS FLOAT) ELSE NULL END) as avg_position_first_half,

    -- Feature 3: Derived CTR (Days 1-15)
    SUM(CASE WHEN EXTRACT(DAY FROM CAST(d.report_date AS DATE)) <= 15 THEN CAST(d.{col_clk} AS FLOAT) ELSE 0 END) /
        NULLIF(SUM(CASE WHEN EXTRACT(DAY FROM CAST(d.report_date AS DATE)) <= 15 THEN CAST(d.{col_imp} AS FLOAT) ELSE 0 END), 0) as derived_ctr_first_half,

    -- Feature 4: Derived Engagement Rate (Days 1-15, strictly checking GA4 availability and using real ga4 columns)
    AVG(CASE WHEN EXTRACT(DAY FROM CAST(d.report_date AS DATE)) <= 15 AND d.ga4_data_available IS TRUE THEN (CAST(d.ga4_engaged_sessions AS FLOAT) / NULLIF(CAST(d.ga4_sessions AS FLOAT), 0)) ELSE NULL END) as engagement_rate_first_half,

    -- Feature 5: Content Age (using real content_created_date from dim_content)
    MAX(DATE_DIFF('day', CAST(c.{col_date} AS DATE), CAST(d.report_date AS DATE))) as content_age_days,

    -- Target Construct components (Days 16-31, specifically for Part C)
    AVG(CASE WHEN EXTRACT(DAY FROM CAST(d.report_date AS DATE)) > 15 AND d.gsc_avg_position > 0 THEN CAST(d.gsc_avg_position AS FLOAT) ELSE NULL END) as avg_position_second_half

FROM '{daily_path}' d
LEFT JOIN '{content_path}' c ON d.{col_content_id} = c.{col_content_id}
GROUP BY d.{col_content_id}
HAVING impressions_first_half > 0
"""
features_df = con.execute(q_features).df()

print(f"Successfully generated {len(features_df):,} feature rows.")
print("\nFeature Contract Verification:")
print("impressions_first_half: available at decision moment because it aggregates only days 1-15, strictly prior to our target prediction window.")
print("avg_position_first_half: available at decision moment because it averages search positions from the first 15 days only.")
print("derived_ctr_first_half: available at decision moment because it is computed exclusively from clicks and impressions accumulated in days 1-15.")
print("engagement_rate_first_half: available at decision moment because it computes engagement ratio using valid GA4 sessions logged before day 16.")
print("content_age_days: available at decision moment because content_created_date is static metadata known entirely prior to the target prediction window.")


# ===== PART C: THE DELIBERATE LEAKAGE TRAP =====
print("\n===== PART C: THE DELIBERATE LEAKAGE TRAP =====")

# 1. Define Binary Proxy Label
# Drop rows where we lack position data in either half, preventing synthetic calculations
model_df = features_df.dropna(subset=['avg_position_first_half', 'avg_position_second_half']).copy()

# Label: 1 if average position got numerically lower (improved) in the second half of the month
model_df['position_improved'] = (model_df['avg_position_second_half'] < model_df['avg_position_first_half']).astype(int)

print("1. Label definition: 'position_improved' = 1 if avg_position_second_half < avg_position_first_half.")
print("   Constructed using report_date windowing within month=2026-03: comparing feature window (days 1-15) vs target window (days 16-31).")

# Clean data for modeling
honest_features = [
    'impressions_first_half',
    'avg_position_first_half',
    'derived_ctr_first_half',
    'engagement_rate_first_half',
    'content_age_days'
]
model_df = model_df.dropna(subset=honest_features)

X_honest = model_df[honest_features]
y = model_df['position_improved']
X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.2, random_state=42)

# 2. Train Honest Model
clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)
honest_preds = clf.predict_proba(X_test)[:, 1]
honest_auc = roc_auc_score(y_test, honest_preds)

print(f"\n2. Honest Baseline Model trained.")

# 3. Train Leaky Model (adding overlapping target-window feature)
X_leaky = model_df[honest_features + ['avg_position_second_half']]
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leaky, y, test_size=0.2, random_state=42)

clf_leaky = DecisionTreeClassifier(max_depth=5, random_state=42)
clf_leaky.fit(X_train_l, y_train_l)
leaky_preds = clf_leaky.predict_proba(X_test_l)[:, 1]
leaky_auc = roc_auc_score(y_test_l, leaky_preds)

print("\n3. Deliberate Leakage Trap Evaluation:")
print(f"   [HONEST] Model ROC-AUC Score: {honest_auc:.4f}")
print(f"   [LEAKY]  Model ROC-AUC Score: {leaky_auc:.4f}  <--- Visually obvious artificial jump!")

# 4. Explanation
print("\n4. WHY is this specific column leaky?")
print("   The injected feature 'avg_position_second_half' is computed directly from the exact same days 16-31 window used to define our target label ('position_improved').")
print("   Because the model has access to the future position average during training, it is not actually predicting an outcome — it is mathematically reading the answer key.")

# 5. Remove leak and retrain
clf_recovered = DecisionTreeClassifier(max_depth=5, random_state=42)
clf_recovered.fit(X_train, y_train)
recovered_preds = clf_recovered.predict_proba(X_test)[:, 1]
recovered_auc = roc_auc_score(y_test, recovered_preds)

print(f"\n5. Recovered Honest Score (ROC-AUC): {recovered_auc:.4f}")
print("   Confirmed: After dropping the leaky target-window column, the model returns to its true, honest predictive performance.")


===== PART A: THREE VERIFICATION QUERIES =====

--- 1. Grain Check ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Number of content_hash_id appearing under more than one client_hash_id: 0
-> Confirmed: content_hash_id is a safe, globally unique identifier across clients. It can be safely used as the sole grouping key for this project's pooled-across-clients unit of analysis.

--- 2. Row Count & Date Span ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total row count for month=2026-03: 9,841,378
Global date span: 2026-03-01 00:00:00 to 2026-03-31 00:00:00

Date span grouped by client_hash_id (Sample of widest and narrowest spans):
         client_hash_id   min_date   max_date  span_days
client_73cda7b4e4f265ea 2026-03-01 2026-03-31         30
client_c182d11e4862a37d 2026-03-01 2026-03-31         30
client_f623b01661d4bfe4 2026-03-01 2026-03-31         30
client_8ae2bfb5aa1ffa1e 2026-03-01 2026-03-31         30
client_a2eeb8899886adde 2026-03-01 2026-03-31         30
client_08d2847f24cf89c1 2026-03-01 2026-03-31         30
client_86ebc2f12c01f586 2026-03-03 2026-03-31         28
client_f6f0cdf26d03d7bd 2026-03-19 2026-03-31         12
client_810019792c9b8efc 2026-03-20 2026-03-31         11
client_e00b29e582949543 2026-03-23 2026-03-31          8
-> Confirmed: History depth varies heavily by client. Global limits obscure truncated client data.

--- 3. Availability Check (GA4 Zero-Fill Audit) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows before filter: 9,841,378
Total rows after 'ga4_data_available IS TRUE' filter: 413,966.0
Survival percentage: 4.21%

===== PART B: FIVE-FEATURE FRAME =====


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Successfully generated 151,981 feature rows.

Feature Contract Verification:
impressions_first_half: available at decision moment because it aggregates only days 1-15, strictly prior to our target prediction window.
avg_position_first_half: available at decision moment because it averages search positions from the first 15 days only.
derived_ctr_first_half: available at decision moment because it is computed exclusively from clicks and impressions accumulated in days 1-15.
engagement_rate_first_half: available at decision moment because it computes engagement ratio using valid GA4 sessions logged before day 16.
content_age_days: available at decision moment because content_created_date is static metadata known entirely prior to the target prediction window.

===== PART C: THE DELIBERATE LEAKAGE TRAP =====
1. Label definition: 'position_improved' = 1 if avg_position_second_half < avg_position_first_half.
   Constructed using report_date windowing within month=2026-03: comparing feature 

## 4. Data limits

This slice cannot support certain claims regardless of modeling choices. GA4 availability is the most severe limitation found: only 4.21 percent of rows in month=2026-03 carry valid GA4 data under the ga4_data_available flag. This gap is structural, not random, tied to when and how completely each client's GA4 tracking was active, and it means any archetype or recommendation built on engagement signals rests on a small, non-representative fraction of this slice. This should be disclosed prominently alongside any engagement-based recommendation, not treated as a minor caveat.

The half-month feature and label split used here is a within-month proxy, not a true trailing or forward window. Splitting month=2026-03 into days 1 through 15 versus 16 through 31 approximates a prev30/last30 pattern but compresses both windows to roughly two weeks each, shorter and noisier than a genuine 30-day trailing window would be. Position and CTR estimates over 15 days carry more volatility than estimates over 30, so this contract's features are noisier than a full monthly aggregate would be.

Filtering out rows with zero first-half impressions also excludes low-signal content by design, reducing the working set from 9,841,378 raw rows down to 151,981 content-level feature rows. This keeps CTR well-defined, but it means this slice systematically leaves out the lowest-visibility content, which is exactly the segment a prune-style recommendation might most need to evaluate. This is a scope limitation to disclose, not a data quality flaw.

Finally, this data has no semantic or textual signal at any point. It can establish that a content item's observed performance changed, but it cannot explain why in content terms, since no keywords, titles, or text exist in this warehouse release. Any archetype this project surfaces describes behavioral and performance patterns, not content-level causes, and output language must stay in observed, measured, and directional terms rather than causal ones.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.